In [70]:
import pandas as pd
DATA = 'data/sample_with_uxagent_runs_2.csv'
sims = pd.read_csv(DATA)
sims

,starting_url,run_folder,persona_name,persona_description,persona_id,flow_name,flow_description,flow_id,category,name
0,https://www.barclays.co.uk/,runs/2025-12-01_14-59-28_6426,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Get customer support number,Get the number for customer service. Stop once...,d08c86b0-a015-46f4-9271-bcc6b02b8fd8,financial,Sim 73
1,https://www.nike.com/gb/,runs/2025-12-01_15-13-11_e441,First-time user,You are a first-time user with no prior experi...,1e501ffd-b1bf-414b-9c7e-493ee3c539ca,Find ladies cotton trousers on sale,Find a pair of trousers or joggers that contai...,a8369faf-4b1b-4ebe-b690-2264d23467f9,fashion,Sim 18
2,https://www.primark.com/,runs/2025-12-01_15-32-04_6b22,Experienced user,You are an experienced user who has used this ...,774d242c-3d09-4102-84e1-602f488f385f,Find men's trainers,Find a pair of trainers in men's size 10 in a ...,4307f7ef-acc1-41bd-8437-08b7f0ecccd8,fashion,Sim 31
3,https://www.santander.co.uk/,runs/2025-12-01_15-53-53_20e5,Generic user,A generic user.,cb5ae8b6-d961-4455-b3e8-eba75c70cd25,Find a credit card,Find a credit card that suits your needs. Stop...,7b9f4a1b-15d1-42d3-a578-a71e4d428e9a,financial,Sim 141
4,https://www.zara.com/uk/,runs/2025-12-01_14-44-15_8a34,Older user,You are a retiree in their 70s. You have typic...,4169e998-48c6-4697-98d1-0113b448a8c5,Find returns policy,Find the returns policy. Stop once you are sat...,233e9131-9020-4254-9157-50013424a695,fashion,Sim 56
5,https://www.nationwide.co.uk/,runs/2025-12-01_16-36-42_e360,First-time user,You are a first-time user with no prior experi...,1e501ffd-b1bf-414b-9c7e-493ee3c539ca,Locate a nearby branch,Find a branch that's near WC1E 6AE. Stop once ...,f0caf61d-07d0-40f5-9115-c383f7481a8f,financial,Sim 110
6,https://www.hsbc.co.uk/,runs/2025-12-01_16-39-54_9fdc,First-time user,You are a first-time user with no prior experi...,1e501ffd-b1bf-414b-9c7e-493ee3c539ca,Find a credit card,Find a credit card that suits your needs. Stop...,7b9f4a1b-15d1-42d3-a578-a71e4d428e9a,financial,Sim 78
7,https://www.next.co.uk/,runs/2025-12-01_16-50-45_edad,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Find ladies cotton trousers on sale,Find a pair of trousers or joggers that contai...,a8369faf-4b1b-4ebe-b690-2264d23467f9,fashion,Sim 12
8,https://www.lloydsbank.com/,runs/2025-12-01_17-04-15_7a81,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Locate a nearby branch,Find a branch that's near WC1E 6AE. Stop once ...,f0caf61d-07d0-40f5-9115-c383f7481a8f,financial,Sim 104
9,https://www.natwest.com/,runs/2025-12-01_17-13-33_a270,Young professional user,You are a young professional in your mid-20s. ...,0a232797-6024-44e4-ae13-6f3b375b10b5,Find a credit card,Find a credit card that suits your needs. Stop...,7b9f4a1b-15d1-42d3-a578-a71e4d428e9a,financial,Sim 132


In [71]:
import json
import os
import anthropic

client = anthropic.Anthropic()

def count_tokens(text):
    response = client.messages.count_tokens(
    model="claude-sonnet-4-5",
    messages=[{
        "role": "user",
        "content": text
    }],
)

    return json.loads(response.model_dump_json())['input_tokens']

def get_text_from_run(run_folder):
    # get the action_trace.json, basic_info.json, and memory.json files from the run folder
    # concat it
    # return the text
    with open(run_folder + '/action_trace.json', 'r') as f:
        action_trace = json.load(f)
    with open(run_folder + '/basic_info.json', 'r') as f:
        basic_info = json.load(f)
    with open(run_folder + '/memory_trace.json', 'r') as f:
        memory = json.load(f)

    # get the simplified HTML from all the files in the run_folder/simp_html folder
    simplified_html = ''
    for file in os.listdir(run_folder + '/simp_html'):
        with open(run_folder + '/simp_html/' + file, 'r') as f:
            simplified_html += f.read()

    output= str(action_trace) + str(basic_info) + str(memory) + str(simplified_html)
    
    print(f'{run_folder}: {count_tokens(output)}')
    
    return output
    
sims['text'] = sims.apply(lambda x: get_text_from_run(x['run_folder']), axis=1)

sims['text_len'] = sims['text'].apply(lambda x: len(x))
sims['input_tokens'] = sims['text'].apply(lambda x: count_tokens(x))
# 

runs/2025-12-01_14-59-28_6426: 80015
runs/2025-12-01_15-13-11_e441: 116164
runs/2025-12-01_15-32-04_6b22: 226459
runs/2025-12-01_15-53-53_20e5: 130764
runs/2025-12-01_14-44-15_8a34: 80565
runs/2025-12-01_16-36-42_e360: 31723
runs/2025-12-01_16-39-54_9fdc: 97591
runs/2025-12-01_16-50-45_edad: 388512
runs/2025-12-01_17-04-15_7a81: 64328
runs/2025-12-01_17-13-33_a270: 73208


In [69]:
count_tokens(""""
<SYSTEM_CAPABILITY> 
* You are a UI/UX researcher who is reviewing a user's experience with a website. You are given a list of past messages and screenshots from the user.
* Markdown the user simulation in 1-2 sentences; concisely say what the user did and what they felt with all the important bits (e.g. "User did xyz.").
* Consider if they completed the ENTIRE task/user flow they were given.
* output a json with summary and success (ie if they completed the flow successfully or not)
* Be concise. Up to 2 sentences for the summary and 1-3 bullet points for the pain points and actionable improvements.
</SYSTEM_CAPABILITY> 
<IMPORTANT> 
Output in the following way.
{{
    "summary": "1-2 sentence summary: explain the flow of the users (ie navigated from x to y to z) and mention any friction points they faced.",
    "success": True or False,
    "recommendations": {{
    "product": "Return a list of up to 3 JSONs. Include a concise explanation of a relevant issue and recommendation, and a short 2–3 word description of a UX-level recommendation. Focus on **strategic, product-level observations** that connect directly to the user’s goals, needs, and frustrations during this simulation — not speculative or sweeping redesigns. These should concern the product’s value proposition, content clarity, feature completeness, or business relevance. Your suggestions should answer **'WHAT should we improve or clarify in the offering?'** or **'WHY might this product not yet meet the user's intent?'**, not **'HOW should the UI change?'** or **'WHAT new audience should we serve?'**.  Be realistic and proportionate: do **not** recommend major product pivots, adding entire new verticals, or front-page changes based on a single user flow. Instead, ground suggestions in observed user evidence .  Format: [{{'description':'<Description 1>','recommendation':'<Emoji + Recommendation 1>','priority':'high|medium|low'}},{{'description':'<Description 2>','recommendation':'<Emoji + Recommendation 2>','priority':'high|medium|low'}}]",
    "ux": "Return a list of up to 3 JSONs. Include a concise explanation of a relevant issue and recommendation, and a short 2–3 word description of a UX-level recommendation. Focus only on user journey, flow logic, decision-making clarity, emotional friction (e.g. based on [emotion] expressed), confidence, or cognitive load. Format: [{{'description':'<Description 1>','recommendation':'<Emoji + Recommendation 1>','priority':'high|medium|low'}},{{'description':'<Description 2>','recommendation':'<Emoji + Recommendation 2>','priority':'high|medium|low'}}]",
    "ui": "Return a list of up to 3 JSONs. Include a concise explanation of a relevant issue and recommendation, and a short 2–3 word description of a UI-level recommendation.  Focus only on layout, visual hierarchy, input behavior, responsiveness, spacing, or styling. Format: [{{'description':'<Description 1>','recommendation':'<Emoji + Recommendation 1>','priority':'high|medium|low'}},{{'description':'<Description 2>','recommendation':'<Emoji + Recommendation 2>','priority':'high|medium|low'}}]"
    }}
}}
Be extremely accurate and always make reference to real user steps. 
Make sure the recommendations are really useful, EXTREMELY DIVERSE and appropriate and helpful for the website given the flow and user. They should be highly realistic and reasonable.
Do not recommend removing cookie pop-ups as they are legally required.
Use (professional) emojis in the recommendations json keys eg ✅ ❌ ⚠️ for statuses 🧭 🔍 🧱 for usability recommendations types 🏠 → 🧾 → ✅ for flow paths.
</IMPORTANT>
""")

942

In [73]:
sims.to_csv('data/sample_uxagent_runs_3.csv', index=False)